# TP2 — Préparation et validation des données
Pipeline (scripts dans `src/`, lancés depuis la racine) :
1. `select_for_annotation.py` → `image_labels.py` → annotation texte (`data/annotated/`)
2. `text_preprocessing.py` → `image_preprocessing.py` → `data/processed/cyberbullying_processed.csv`
3. `split_data.py` → `train.csv`, `validation.csv`, `test.csv`
4. `rebalance.py` → `train_balanced.csv` (train uniquement)

Ce notebook : **analyse** (Partie 3) et contrôle des résultats.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
df = pd.read_csv(PROC / "cyberbullying_processed.csv")
df["image_path"] = df["image_path"].fillna("")
df["has_image"] = df["image_path"] != ""
df["n_words"] = df["text"].fillna("").str.split().str.len()

BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e6e5e0"
def style(ax, title):
    ax.set_title(title, loc="left", color=INK, fontsize=12)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(colors=MUTED); ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)

## Partie 3.7 — Analyse descriptive

In [ ]:
desc = {
    "publications annotées": len(df),
    "images disponibles": int(df["has_image"].sum()),
    "texte seul": int((~df["has_image"]).sum()),
    "texte + image": int(df["has_image"].sum()),
}
print(pd.Series(desc).to_string())
print("\nPublications par classe :"); print(df["label"].value_counts().to_string())
print("\nClasses selon la source :"); print(pd.crosstab(df["label"], df["source"], margins=True).to_string())
print("\nLongueur moyenne (mots) par classe :"); print(df.groupby("label")["n_words"].mean().round(1).sort_values().to_string())

In [ ]:
counts = df["label"].value_counts().sort_values()
length = df.groupby("label")["n_words"].mean().reindex(counts.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(counts.index, counts.values, color=BLUE, height=0.6)
for y, v in enumerate(counts.values):
    axes[0].text(v + 3, y, f"{v} ({v / len(df):.0%})", va="center", color=MUTED, fontsize=9)
style(axes[0], "Publications par classe")
axes[1].barh(length.index, length.values, color=BLUE, height=0.6)
for y, v in enumerate(length.values):
    axes[1].text(v + 1, y, f"{v:.0f}", va="center", color=MUTED, fontsize=9)
style(axes[1], "Longueur moyenne (mots)")
axes[1].set_yticklabels([])
plt.tight_layout(); fig.savefig(REPORTS / "tp2_class_distribution.png", dpi=120); plt.show()

## Partie 3.8 — Déséquilibre des classes

In [ ]:
prop = df["label"].value_counts(normalize=True)
imb = pd.DataFrame({"n": df["label"].value_counts(), "proportion": prop.map("{:.1%}".format)})
print(imb.to_string())
print(f"\nmajoritaire : {prop.idxmax()} — minoritaire : {prop.idxmin()}")
print(f"ratio de déséquilibre (max / min) : {df['label'].value_counts().max() / df['label'].value_counts().min():.0f}")

**Conséquences sur l'apprentissage**
- Un modèle qui prédit toujours `non_toxic` obtient déjà ~52 % d'accuracy → l'accuracy est trompeuse : on suivra le **F1 macro** (TP3).
- Les classes minoritaires (surtout `threat_intimidation`, 9 exemples) seront peu apprises : le modèle a peu d'exemples et la perte est dominée par la classe majoritaire.
- Avec si peu d'exemples, la validation et le test ne contiennent que 1 à 2 menaces : leurs scores sur cette classe seront très instables.
- Réponses : rééquilibrage du **train uniquement** (Partie 3.9) et/ou pondération des classes dans la perte (TP3).

## Partie 4 — Contrôle du découpage

In [ ]:
splits = {k: pd.read_csv(PROC / f"{k}.csv") for k in ["train", "validation", "test"]}
dist = pd.DataFrame({k: v["label"].value_counts(normalize=True) for k, v in splits.items()})
print((dist * 100).round(1).to_string())   # proportions (%) proches entre les 3 ensembles

## Partie 3.9 — Rééquilibrage (train uniquement)
Sous-échantillonnage de `non_toxic` (→ 150) + sur-échantillonnage des minoritaires (→ 100, au plus ×5 copies).
Validation et test gardent la distribution réelle ; `train.csv` d'origine est conservé.

In [ ]:
bal = pd.read_csv(PROC / "train_balanced.csv")
comp = pd.DataFrame({"avant": splits["train"]["label"].value_counts(),
                     "après": bal["label"].value_counts()}).sort_values("avant")
print(comp.to_string())

ORANGE = "#eb6834"
fig, ax = plt.subplots(figsize=(8, 4))
y = range(len(comp)); h = 0.38
ax.barh([i + h / 2 for i in y], comp["avant"], height=h, color=BLUE, label="avant")
ax.barh([i - h / 2 for i in y], comp["après"], height=h, color=ORANGE, label="après")
ax.set_yticks(list(y)); ax.set_yticklabels(comp.index)
ax.legend(frameon=False, loc="lower right")
style(ax, "Train : distribution avant / après rééquilibrage")
plt.tight_layout(); fig.savefig(REPORTS / "tp2_rebalancing.png", dpi=120); plt.show()

**Limite** : les 30 menaces du train ne sont que 6 textes copiés 5 fois → risque de sur-apprentissage.
En TP3, comparer avec la pondération des classes dans la perte (sans duplication).